# MentalDistress Phase 9 — Routing Calibration (GPU)

Phase 9 calibrates only the routing threshold `tau` on the locked validation split. The
Phase 5 hard-pair set and `K=3` remain frozen, and all 100 Phase 8 specialist heads remain
frozen. The test split is never discovered, loaded, or evaluated.

## 1. Environment and frozen configuration

This cell initializes the offline Kaggle GPU runtime and declares every frozen constant in
one visible place. `tau=0.00` is the global-model-only control.

In [1]:
import gc
import hashlib
import json
import os
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from safetensors.torch import load_file
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_DIR = WORKING_ROOT / "phase9_routing_calibration_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running Phase 9.")

DEVICE = torch.device("cuda")
SEEDS = [13, 42, 77, 123, 211, 340, 512, 1024, 2026, 3407]
LABELS = ["Suicidal", "Depressed", "Anxious", "Frustrated", "Others"]
LABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}

K_FIXED = 3
TAU_GRID = np.array(
    [0.00, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.70, 0.90, 1.00]
)
PHASE8 = "phase8_all_pair_specialists_results"
PHASE9 = "phase9_routing"

TRAIN_SHA256 = "1bac0d2f4e0f4d3da940b08a21bd3c6e18b1d958c57cb263c89a900381572d5b"
VALIDATION_SHA256 = "88835cd714b02efbd230536c7de3f455e807051a23a1786640b0e8b26b430165"

print("CUDA device:", torch.cuda.get_device_name(0))
print("Seeds:", SEEDS)
print("Fixed K:", K_FIXED)
print("Tau grid:", TAU_GRID.tolist())
print("Test data accessed: False")

CUDA device: Tesla T4
Seeds: [13, 42, 77, 123, 211, 340, 512, 1024, 2026, 3407]
Fixed K: 3
Tau grid: [0.0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9, 1.0]
Test data accessed: False


## 2. Required Kaggle inputs

Attach four inputs before execution:

1. The locked MentalDistress dataset containing `MentalDistress_validation_LOCKED.csv`.
2. The frozen Phase 5 bootstrap-stability results.
3. `MentalDistress_Phase8_All_Pair_Specialists_Results.zip` or its extracted directory.
4. The ten complete MentalRoBERTa global-model run directories and their summary CSV.

The helper functions below discover files recursively, reject unsafe ZIP members, and
verify every supplied SHA-256 manifest before model inference begins.

In [2]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def safe_extract(archive_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member_name in archive.namelist():
            member_path = (destination / member_name).resolve()
            expected_prefix = str(destination) + os.sep
            if not str(member_path).startswith(expected_prefix):
                raise ValueError(f"Unsafe ZIP member: {member_name}")
        archive.extractall(destination)


def verify_manifest(root):
    manifest_path = root / "sha256_manifest.csv"
    manifest_frame = pd.read_csv(manifest_path)
    actual_paths = {
        path.relative_to(root).as_posix()
        for path in root.rglob("*")
        if path.is_file() and path != manifest_path
    }
    expected_paths = set(manifest_frame["path"])
    if expected_paths != actual_paths:
        raise ValueError(f"Manifest membership mismatch: {root}")

    for record in manifest_frame.itertuples():
        file_path = root / record.path
        size_matches = file_path.is_file() and file_path.stat().st_size == int(record.bytes)
        hash_matches = file_path.is_file() and sha256_file(file_path) == record.sha256
        if not size_matches or not hash_matches:
            raise ValueError(f"Manifest mismatch: {record.path}")


def find_input(filename):
    matches = sorted(INPUT_ROOT.rglob(filename), key=str)
    if not matches:
        raise FileNotFoundError(f"Required Kaggle input not found: {filename}")
    return matches[0]


def metric(y_true, y_predicted):
    return {
        "accuracy": accuracy_score(y_true, y_predicted),
        "macro_f1": f1_score(y_true, y_predicted, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_predicted),
    }


def pair_slug(first_label, second_label):
    return f"{first_label.lower()}__{second_label.lower()}"


print("PASS: integrity and discovery helpers initialized")

PASS: integrity and discovery helpers initialized


## 3. Verify frozen inputs

This gate verifies the locked validation split, the Phase 5 hard-pair decision, and the
complete Phase 8 specialist archive. Phase 9 stops immediately if provenance, membership,
head count, frozen `H`, or the no-routing Phase 8 protocol does not match.

In [3]:
validation_path = find_input("MentalDistress_validation_LOCKED.csv")
if sha256_file(validation_path) != VALIDATION_SHA256:
    raise ValueError("Locked-validation SHA-256 mismatch")

validation = pd.read_csv(validation_path)
if len(validation) != 1490:
    raise ValueError(f"Expected 1490 validation rows, found {len(validation)}")
if set(validation["label"]) != set(LABELS):
    raise ValueError("Locked-validation labels do not match the frozen label set")

phase5_selection_path = find_input("phase5_hard_pair_selection.json")
phase5_dir = phase5_selection_path.parent
verify_manifest(phase5_dir)
phase5_selection = json.loads(phase5_selection_path.read_text())

HARD_PAIRS = phase5_selection["H"]
expected_hard_pairs = [
    "Frustrated|Others",
    "Suicidal|Depressed",
    "Depressed|Others",
]
assert phase5_selection["status"] == "FROZEN"
assert phase5_selection["K"] == K_FIXED
assert HARD_PAIRS == expected_hard_pairs
assert phase5_selection["test_accessed"] is False

phase8_archives = sorted(
    INPUT_ROOT.rglob("MentalDistress_Phase8_All_Pair_Specialists_Results.zip"),
    key=str,
)
phase8_directories = sorted(
    [path for path in INPUT_ROOT.rglob(PHASE8) if path.is_dir()],
    key=str,
)
phase8_cache = WORKING_ROOT / "phase8_input"

if phase8_directories:
    phase8_dir = phase8_directories[0]
elif phase8_archives:
    if phase8_cache.exists():
        shutil.rmtree(phase8_cache)
    phase8_cache.mkdir(parents=True)
    safe_extract(phase8_archives[0], phase8_cache)
    phase8_dir = phase8_cache / PHASE8
else:
    raise FileNotFoundError("Attach the Phase 8 results ZIP or extracted directory")

verify_manifest(phase8_dir)
phase8_protocol = json.loads((phase8_dir / "phase8_protocol.json").read_text())
assert phase8_protocol["head_count"] == 100
assert phase8_protocol["routing_performed"] is False

print("Validation:", validation_path)
print("Phase 5:", phase5_dir)
print("Phase 8:", phase8_dir)
print("Frozen hard pairs:", HARD_PAIRS)
print("PASS: locked validation, Phase 5, and all 100 Phase 8 heads verified")

Validation: /kaggle/input/datasets/sayemahmedshayeed/mentaldistress-locked-dataset/MentalDistress_validation_LOCKED.csv
Phase 5: /kaggle/input/datasets/sumitnath07/mentaldistress-phase5-bootstrap-stability-results/phase5_bootstrap_stability_results
Phase 8: /kaggle/input/datasets/sumitnath07/mentaldistress-phase8-all-pair-specialists-results/phase8_all_pair_specialists_results
Frozen hard pairs: ['Frustrated|Others', 'Suicidal|Depressed', 'Depressed|Others']
PASS: locked validation, Phase 5, and all 100 Phase 8 heads verified


## 4. Load and verify global checkpoints

Each specialist must be paired with the global checkpoint from the same seed. This cell
resolves the ten canonical run IDs from the frozen summary and verifies both model and
validation-artifact manifests before any checkpoint is loaded onto the GPU.

In [4]:
summary_path = find_input("mentalroberta_full_10seed_summary.csv")
global_summary = pd.read_csv(summary_path)
summary_seeds = sorted(global_summary["seed"].astype(int).tolist())
if summary_seeds != SEEDS:
    raise ValueError(f"Canonical global seeds mismatch: {summary_seeds}")

global_runs = {}
for record in global_summary.itertuples():
    seed = int(record.seed)
    candidates = [
        path
        for path in INPUT_ROOT.rglob(str(record.run_id))
        if path.is_dir()
        and (path / "models/best_model/model.safetensors").is_file()
        and (path / "artifacts/validation_predictions.csv").is_file()
    ]
    if not candidates:
        raise FileNotFoundError(f"Complete global run not found: {record.run_id}")

    run_dir = sorted(candidates, key=str)[0]
    verify_manifest(run_dir / "models")
    verify_manifest(run_dir / "artifacts")
    global_runs[seed] = run_dir
    print(f"  seed {seed}: {run_dir.name}")

if sorted(global_runs) != SEEDS:
    raise ValueError("Did not resolve exactly one canonical run for every seed")

print("PASS: ten seed-matched global checkpoints verified")

  seed 13: mentalroberta_full_seed13_len512_fea2991778f3
  seed 42: mentalroberta_full_seed42_len512_be72f916ca15
  seed 77: mentalroberta_full_seed77_len512_d14375990239
  seed 123: mentalroberta_full_seed123_len512_d1ad313ee75d
  seed 211: mentalroberta_full_seed211_len512_149308113b40
  seed 340: mentalroberta_full_seed340_len512_6353292c8230
  seed 512: mentalroberta_full_seed512_len512_412e1caa7392
  seed 1024: mentalroberta_full_seed1024_len512_e91f034d8a71
  seed 2026: mentalroberta_full_seed2026_len512_f6630d942394
  seed 3407: mentalroberta_full_seed3407_len512_d8596ea710c3
PASS: ten seed-matched global checkpoints verified


## 5. Frozen inference helpers

The global model supplies two frozen quantities for each validation row: class
probabilities and the final-layer token-zero representation. A Phase 8 linear specialist
consumes that representation only when the routing gate selects its label pair.

In [5]:
def extract_features_and_probabilities(run_dir, batch_size=16):
    model_dir = run_dir / "models/best_model"
    tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_dir,
        local_files_only=True,
        output_hidden_states=True,
    )
    model = model.eval().to(DEVICE)

    feature_batches = []
    probability_batches = []
    texts = validation["text"].astype(str).tolist()

    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            encoded = tokenizer(
                texts[start : start + batch_size],
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            )
            encoded = {name: tensor.to(DEVICE) for name, tensor in encoded.items()}
            output = model(
                **encoded,
                output_hidden_states=True,
                return_dict=True,
            )
            feature_batches.append(output.hidden_states[-1][:, 0, :].float().cpu())
            probabilities = torch.softmax(output.logits, dim=1).float().cpu().numpy()
            probability_batches.append(probabilities)

    del model, tokenizer
    torch.cuda.empty_cache()
    features = torch.cat(feature_batches)
    probabilities = np.vstack(probability_batches)
    return features, probabilities


def load_specialist_head(seed, first_label, second_label, feature_dimension):
    head_dir = (
        phase8_dir
        / "heads"
        / f"seed_{seed}"
        / pair_slug(first_label, second_label)
    )
    metadata = json.loads((head_dir / "metadata.json").read_text())
    expected_pair = f"{first_label}|{second_label}"
    if metadata["pair"] != expected_pair:
        raise ValueError(f"Specialist identity mismatch: {head_dir}")

    tensors = load_file(str(head_dir / "head.safetensors"), device="cpu")
    weight = tensors["classifier.weight"]
    bias = tensors["classifier.bias"]
    if weight.shape[1] != feature_dimension:
        raise ValueError(f"Specialist feature dimension mismatch: {head_dir}")
    return weight, bias


print("PASS: frozen inference helpers initialized")

PASS: frozen inference helpers initialized


## 6. Calibrate the routing threshold

For every seed and every candidate `tau`, routing requires both conditions:

- the global model's unordered top-two labels equal one frozen hard pair; and
- the top-two probability margin is at most `tau`.

At `tau=0.00`, the routing loop is bypassed explicitly, including for exact probability
ties. This makes the first grid row a genuine global-model-only control.

In [6]:
routing_rows = []
routing_details = {}
y_true = validation["label"].map(LABEL_TO_ID).to_numpy()

for seed_index, seed in enumerate(SEEDS, start=1):
    print(f"[{seed_index}/{len(SEEDS)}] seed {seed}: frozen global inference")
    features, probabilities = extract_features_and_probabilities(global_runs[seed])

    top2 = np.argsort(-probabilities, axis=1)[:, :2]
    row_indices = np.arange(len(y_true))
    margin = (
        probabilities[row_indices, top2[:, 0]]
        - probabilities[row_indices, top2[:, 1]]
    )
    global_prediction = top2[:, 0]

    specialist_predictions = {}
    for pair_name in HARD_PAIRS:
        first_label, second_label = pair_name.split("|")
        weight, bias = load_specialist_head(
            seed,
            first_label,
            second_label,
            features.shape[1],
        )
        logits = features.numpy() @ weight.numpy().T + bias.numpy()
        specialist_predictions[pair_name] = np.where(
            logits.argmax(axis=1) == 0,
            LABEL_TO_ID[first_label],
            LABEL_TO_ID[second_label],
        )

    for tau in TAU_GRID:
        final_prediction = global_prediction.copy()
        routed = np.zeros(len(y_true), dtype=bool)
        routed_pair = np.full(len(y_true), "", dtype=object)

        if not np.isclose(tau, 0.0):
            for pair_name in HARD_PAIRS:
                first_label, second_label = pair_name.split("|")
                pair_ids = {LABEL_TO_ID[first_label], LABEL_TO_ID[second_label]}
                pair_matches = np.array(
                    [set(row_top2) == pair_ids for row_top2 in top2]
                )
                gate = pair_matches & (margin <= tau)
                final_prediction[gate] = specialist_predictions[pair_name][gate]
                routed |= gate
                routed_pair[gate] = pair_name

        metrics = metric(y_true, final_prediction)
        corrections = int(
            (routed & (global_prediction != y_true) & (final_prediction == y_true)).sum()
        )
        harms = int(
            (routed & (global_prediction == y_true) & (final_prediction != y_true)).sum()
        )

        if np.isclose(tau, 0.0):
            baseline_metrics = metric(y_true, global_prediction)
            assert not routed.any()
            assert np.array_equal(final_prediction, global_prediction)
            assert corrections == 0 and harms == 0
            assert all(
                np.isclose(metrics[key], baseline_metrics[key])
                for key in baseline_metrics
            )

        routing_rows.append(
            {
                "model_seed": seed,
                "tau": float(tau),
                **metrics,
                "routing_coverage": routed.mean(),
                "routed_n": int(routed.sum()),
                "corrections": corrections,
                "harms": harms,
                "is_no_routing_baseline": bool(np.isclose(tau, 0.0)),
            }
        )
        routing_details[(seed, float(tau))] = (
            final_prediction,
            routed,
            routed_pair,
            global_prediction,
            top2,
            margin,
        )

    del features, probabilities
    gc.collect()

print("PASS: evaluated every seed and tau configuration")

[1/10] seed 13: frozen global inference


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[2/10] seed 42: frozen global inference


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[3/10] seed 77: frozen global inference


...

[9/10] seed 2026: frozen global inference


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[10/10] seed 3407: frozen global inference


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PASS: evaluated every seed and tau configuration


## 7. Validate and select the configuration

First, the notebook proves that all ten `tau=0.00` rows have zero coverage, zero routed
examples, zero corrections, and zero harms. It then aggregates seeds equally and selects
the greatest mean validation macro-F1. Candidates within 0.001 of that maximum prefer
lower coverage and then lower `tau`, allowing no routing to remain the final choice.

In [7]:
seedwise_results = pd.DataFrame(routing_rows)
zero_rows = seedwise_results[np.isclose(seedwise_results["tau"], 0.0)]

assert len(zero_rows) == len(SEEDS)
assert zero_rows["routed_n"].eq(0).all()
assert np.allclose(zero_rows["routing_coverage"], 0.0)
assert zero_rows["corrections"].eq(0).all()
assert zero_rows["harms"].eq(0).all()
print("PASS: tau=0.00 is an exact no-routing baseline")

routing_grid = seedwise_results.groupby("tau", as_index=False).agg(
    mean_macro_f1=("macro_f1", "mean"),
    sd_macro_f1=("macro_f1", "std"),
    mean_accuracy=("accuracy", "mean"),
    mean_coverage=("routing_coverage", "mean"),
    mean_corrections=("corrections", "mean"),
    mean_harms=("harms", "mean"),
)

maximum_macro_f1 = routing_grid["mean_macro_f1"].max()
eligible = routing_grid[
    routing_grid["mean_macro_f1"] >= maximum_macro_f1 - 0.001
]
selected_row = eligible.sort_values(["mean_coverage", "tau"]).iloc[0]
BEST_TAU = float(selected_row["tau"])
selected_mode = "no_routing" if np.isclose(BEST_TAU, 0.0) else "specialist_routing"

print("Selected tau:", BEST_TAU)
print("Selected mode:", selected_mode)
display(routing_grid.round(6))

PASS: tau=0.00 is an exact no-routing baseline
Selected tau: 0.1
Selected mode: specialist_routing


,tau,mean_macro_f1,sd_macro_f1,mean_accuracy,mean_coverage,mean_corrections,mean_harms
0,0.00,0.863198,0.003567,0.866577,0.000000,0.0,0.0
1,0.05,0.863726,0.003934,0.867114,0.002953,1.4,0.6
2,0.10,0.864138,0.004309,0.867517,0.006577,2.6,1.2
3,0.20,0.864079,0.004422,0.867450,0.013826,3.6,2.3
4,0.30,0.864859,0.004436,0.868255,0.021342,5.6,3.1
5,0.40,0.864625,0.004353,0.868054,0.029060,6.3,4.1
6,0.50,0.864688,0.004376,0.868121,0.037584,6.9,4.6
7,0.70,0.864909,0.003855,0.868255,0.062819,7.8,5.3
8,0.90,0.864993,0.003989,0.868322,0.114966,8.3,5.7
9,1.00,0.864921,0.004024,0.868255,0.556577,8.3,5.8


## 8. Export verified results

The selected configuration is exported separately for every seed. Each prediction file
retains the global decision, final decision, routing indicator, chosen pair, probability
margin, and original locked-validation row identifier. Confusion matrices and class reports
use the frozen five-label order.

In [8]:
for seed in SEEDS:
    (
        final_prediction,
        routed,
        routed_pair,
        global_prediction,
        top2,
        margin,
    ) = routing_details[(seed, BEST_TAU)]

    prediction_frame = pd.DataFrame(
        {
            "original_row": validation["original_row"],
            "true_label": validation["label"],
            "global_id": global_prediction,
            "final_id": final_prediction,
            "routed": routed,
            "routed_pair": routed_pair,
            "margin": margin,
            "top1": top2[:, 0],
            "top2": top2[:, 1],
        }
    )
    prediction_frame.to_csv(
        OUTPUT_DIR / f"selected_routing_predictions_seed{seed}.csv",
        index=False,
    )

    confusion = pd.DataFrame(
        confusion_matrix(y_true, final_prediction, labels=range(5)),
        index=LABELS,
        columns=LABELS,
    )
    confusion.to_csv(
        OUTPUT_DIR / f"selected_routing_confusion_matrix_seed{seed}.csv"
    )

    report = classification_report(
        y_true,
        final_prediction,
        labels=range(5),
        target_names=LABELS,
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(report).T.to_csv(
        OUTPUT_DIR / f"selected_routing_report_seed{seed}.csv"
    )

seedwise_results.to_csv(OUTPUT_DIR / "phase9_routing_seedwise.csv", index=False)
routing_grid.to_csv(OUTPUT_DIR / "phase9_routing_grid.csv", index=False)

selected_config = {
    "phase": 9,
    "K_fixed": K_FIXED,
    "H": HARD_PAIRS,
    "tau_grid": TAU_GRID.tolist(),
    "no_routing_baseline_included": True,
    "selected_tau": BEST_TAU,
    "selected_mode": selected_mode,
    "selection": "max mean macro-F1; within 0.001 lower coverage then lower tau",
    "test_accessed": False,
}
config_path = OUTPUT_DIR / "phase9_selected_config.json"
config_path.write_text(json.dumps(selected_config, indent=2) + "\n")

print("PASS: selected predictions, metrics, and reports exported for ten seeds")

PASS: selected predictions, metrics, and reports exported for ten seeds


## 9. Package and execution summary

The final cell creates a complete SHA-256 manifest, verifies it immediately, and writes a
deterministic results ZIP with fixed timestamps. Download the ZIP from Kaggle working
storage and retain its printed SHA-256 digest with the experiment record.

In [9]:
manifest_rows = []
result_files = sorted(path for path in OUTPUT_DIR.rglob("*") if path.is_file())
for result_path in result_files:
    manifest_rows.append(
        {
            "path": result_path.relative_to(OUTPUT_DIR).as_posix(),
            "sha256": sha256_file(result_path),
            "bytes": result_path.stat().st_size,
        }
    )

manifest_path = OUTPUT_DIR / "sha256_manifest.csv"
pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)
verify_manifest(OUTPUT_DIR)

zip_path = WORKING_ROOT / "MentalDistress_Phase9_Routing_Calibration_Results.zip"
with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED,
    compresslevel=9,
) as archive:
    packaged_files = sorted(path for path in OUTPUT_DIR.rglob("*") if path.is_file())
    for result_path in packaged_files:
        archive_name = OUTPUT_DIR.name + "/" + result_path.relative_to(OUTPUT_DIR).as_posix()
        archive_info = zipfile.ZipInfo(archive_name, (2020, 1, 1, 0, 0, 0))
        archive_info.compress_type = zipfile.ZIP_DEFLATED
        archive.writestr(archive_info, result_path.read_bytes())

print("=" * 72)
print("PHASE 9 COMPLETE")
print("Selected tau:", BEST_TAU)
print("Selected mode:", selected_mode)
print("Results ZIP:", zip_path)
print("ZIP SHA256:", sha256_file(zip_path))
print("Test data accessed: False")
print("=" * 72)

PHASE 9 COMPLETE
Selected tau: 0.1
Selected mode: specialist_routing
Results ZIP: /kaggle/working/MentalDistress_Phase9_Routing_Calibration_Results.zip
ZIP SHA256: 9db43d16d0acd7c0fb68129ee2c2e72b88ed3a4ff73c8df7a7ecfd80fcc68ba3
Test data accessed: False
